In [ ]:
!pip install requests beautifulsoup4 pandas --quiet

In [ ]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from typing import Optional
import csv

In [ ]:
BASE_BLOG = "https://www.ruangguru.com/blog"
BASE_API  = "https://www.ruangguru.com/blog/wp-json/wp/v2"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

CATEGORY_URLS = [
    f"{BASE_BLOG}/c/sejarah/sejarah-smp-kelas-{k}" for k in range(7, 10)
] + [
    f"{BASE_BLOG}/c/sejarah/sejarah-sma-kelas-{k}" for k in range(10, 13)
]

print("Target category URLs:")
for url in CATEGORY_URLS:
    print(f"  {url}")

Target category URLs:
  https://www.ruangguru.com/blog/c/sejarah/sejarah-smp-kelas-7
  https://www.ruangguru.com/blog/c/sejarah/sejarah-smp-kelas-8
  https://www.ruangguru.com/blog/c/sejarah/sejarah-smp-kelas-9
  https://www.ruangguru.com/blog/c/sejarah/sejarah-sma-kelas-10
  https://www.ruangguru.com/blog/c/sejarah/sejarah-sma-kelas-11
  https://www.ruangguru.com/blog/c/sejarah/sejarah-sma-kelas-12


In [ ]:
def get_last_page_number(soup) -> int:
    max_page = 1
    for a in soup.find_all("a", href=True):
        match = re.search(r"/page/(\d+)/?$", a["href"])
        if match:
            page_num = int(match.group(1))
            if page_num > max_page:
                max_page = page_num
    return max_page


def extract_slugs_from_soup(soup, category_label: str) -> list:
    results = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if not href.startswith(f"{BASE_BLOG}/"):
            continue
        path = href.replace(BASE_BLOG, "").lstrip("/")
        if any(path.startswith(p) for p in ["c/", "tag/", "#", "?", "page/"]):
            continue
        if not path:
            continue
        results.append({
            "slug":            path.rstrip("/"),
            "url":             href,
            "source_category": category_label,
        })
    return results


def get_all_slugs_from_category(base_category_url: str) -> list:
    category_label = base_category_url.rstrip("/").split("/")[-1]
    all_slugs = {}

    try:
        r = requests.get(base_category_url, headers=HEADERS, timeout=15)
        r.raise_for_status()
    except requests.RequestException as e:
        print(f"Cannot fetch {base_category_url}: {e}")
        return []

    soup = BeautifulSoup(r.text, "html.parser")
    last_page = get_last_page_number(soup)
    print(f"  📄 {category_label}: {last_page} page(s) detected")

    for item in extract_slugs_from_soup(soup, category_label):
        all_slugs[item["slug"]] = item

    # page 2 - n
    for page_num in range(2, last_page + 1):
        page_url = f"{base_category_url}/page/{page_num}"
        print(f"     → Fetching page {page_num}/{last_page}: {page_url}")
        try:
            r = requests.get(page_url, headers=HEADERS, timeout=15)
            r.raise_for_status()
        except requests.RequestException as e:
            print(f"Failed: {e}")
            break

        soup = BeautifulSoup(r.text, "html.parser")
        for item in extract_slugs_from_soup(soup, category_label):
            all_slugs[item["slug"]] = item

        time.sleep(0.8)

    print(f"{len(all_slugs)} unique article(s) from {category_label}")
    return list(all_slugs.values())

In [ ]:
all_articles_seed = []
global_seen_slugs = set()

for cat_url in CATEGORY_URLS:
    articles = get_all_slugs_from_category(cat_url)
    for art in articles:
        if art["slug"] not in global_seen_slugs:
            global_seen_slugs.add(art["slug"])
            all_articles_seed.append(art)
    time.sleep(1.0)

print(f"\nGrand total unique article slugs: {len(all_articles_seed)}")
df_seed = pd.DataFrame(all_articles_seed)
print(df_seed["source_category"].value_counts().to_string())
df_seed.head(10)

  📄 sejarah-smp-kelas-7: 1 page(s) detected
4 unique article(s) from sejarah-smp-kelas-7
  📄 sejarah-smp-kelas-8: 1 page(s) detected
4 unique article(s) from sejarah-smp-kelas-8
  📄 sejarah-smp-kelas-9: 2 page(s) detected
     → Fetching page 2/2: https://www.ruangguru.com/blog/c/sejarah/sejarah-smp-kelas-9/page/2
13 unique article(s) from sejarah-smp-kelas-9
  📄 sejarah-sma-kelas-10: 5 page(s) detected
     → Fetching page 2/5: https://www.ruangguru.com/blog/c/sejarah/sejarah-sma-kelas-10/page/2
     → Fetching page 3/5: https://www.ruangguru.com/blog/c/sejarah/sejarah-sma-kelas-10/page/3
     → Fetching page 4/5: https://www.ruangguru.com/blog/c/sejarah/sejarah-sma-kelas-10/page/4
     → Fetching page 5/5: https://www.ruangguru.com/blog/c/sejarah/sejarah-sma-kelas-10/page/5
42 unique article(s) from sejarah-sma-kelas-10
  📄 sejarah-sma-kelas-11: 5 page(s) detected
     → Fetching page 2/5: https://www.ruangguru.com/blog/c/sejarah/sejarah-sma-kelas-11/page/2
     → Fetching page 3/5: 

,slug,url,source_category
0,kerajaan-samudera-pasai,https://www.ruangguru.com/blog/kerajaan-samude...,sejarah-smp-kelas-7
1,sejarah-nenek-moyang-bangsa-indonesia,https://www.ruangguru.com/blog/sejarah-nenek-m...,sejarah-smp-kelas-7
2,manusia-dan-dinosaurus,https://www.ruangguru.com/blog/manusia-dan-din...,sejarah-smp-kelas-7
3,perang-saudara-kerajaan-kediri,https://www.ruangguru.com/blog/perang-saudara-...,sejarah-smp-kelas-7
4,masa-penjajahan,https://www.ruangguru.com/blog/masa-penjajahan,sejarah-smp-kelas-8
5,kedatangan-nica-di-indonesia,https://www.ruangguru.com/blog/kedatangan-nica...,sejarah-smp-kelas-8
6,kedatangan-pasukan-afnei-di-indonesia,https://www.ruangguru.com/blog/kedatangan-pasu...,sejarah-smp-kelas-8
7,tahukah-kamu-mengenai-organisasi-pergerakan-in...,https://www.ruangguru.com/blog/tahukah-kamu-me...,sejarah-smp-kelas-8
8,sejarah-kelahiran-masa-orde-baru,https://www.ruangguru.com/blog/sejarah-kelahir...,sejarah-smp-kelas-9
9,mengenal-perjanjian-renville,https://www.ruangguru.com/blog/mengenal-perjan...,sejarah-smp-kelas-9


In [ ]:
def fetch_post_by_slug(slug: str) -> Optional[dict]:
    """
    Hits WP REST API with ?slug=<slug>.
    Returns structured dict or None on failure.
    """
    url    = f"{BASE_API}/posts"
    params = {
        "slug":    slug,
        "_fields": "id,slug,title,link,date,content,excerpt",
    }
    try:
        r = requests.get(url, params=params, headers=HEADERS, timeout=15)
        r.raise_for_status()
        data = r.json()
    except requests.RequestException as e:
        print(f"    API error for slug '{slug}': {e}")
        return None

    if not data:
        print(f"    Slug not found in API: '{slug}'")
        return None

    post = data[0]
    return {
        "id":      post.get("id"),
        "slug":    post.get("slug"),
        "title":   post["title"]["rendered"],
        "url":     post.get("link"),
        "date":    post.get("date"),
        "content": post["content"]["rendered"],
        "excerpt": post["excerpt"]["rendered"],
    }


def strip_html(html: str) -> str:
    """
    Cleans WP API content HTML into pure article prose.
    Outputs a single continuous paragraph per article (CSV-safe).
    """
    soup = BeautifulSoup(html, "html.parser")

    NOISE_TAGS = [
        "img", "figure", "figcaption", "picture",
        "video", "audio", "iframe",
        "script", "style", "noscript",
        "button", "form", "input",
        "svg", "canvas",
        "nav", "aside", "footer", "header",
        "advertisement",
    ]
    for tag in soup.find_all(NOISE_TAGS):
        tag.decompose()

    NOISE_PATTERNS = [
        "shortcode", "widget", "banner", "promo", "cta",
        "related", "recommendation", "sidebar", "ads", "advertisement",
        "social-share", "share-button", "newsletter", "popup",
        "breadcrumb", "tag-list", "author-box",
    ]
    for pattern in NOISE_PATTERNS:
        for tag in soup.find_all(
            True,
            class_=lambda c: c and pattern in " ".join(c).lower()
        ):
            tag.decompose()

    BLOCK_TAGS = [
        "p", "div", "h1", "h2", "h3", "h4", "h5", "h6",
        "li", "br", "tr", "blockquote", "hr", "section",
        "ul", "ol", "table", "thead", "tbody",
    ]
    for tag in soup.find_all(BLOCK_TAGS):
        tag.insert_before(" ")
        tag.insert_after(" ")

    text = soup.get_text(separator="")

    text = re.sub(r"[\n\r\t]+", " ", text)    # newlines/tabs → space
    text = re.sub(r" {2,}", " ", text)         # multi-space → single space

    text = re.sub(r"\[.*?\]", "", text)        # remove [caption], [gallery], etc.
    text = re.sub(r" {2,}", " ", text)         # re-collapse after shortcode removal

    return text.strip()

In [ ]:
enriched_records = []
total = len(all_articles_seed)

for i, item in enumerate(all_articles_seed, start=1):
    slug     = item["slug"]
    category = item["source_category"]
    print(f"[{i:>3}/{total}] Fetching: {slug}  ({category})")

    post_data = fetch_post_by_slug(slug)
    if post_data:
        post_data["source_category"] = category
        post_data["content_clean"]   = strip_html(post_data["content"])
        post_data["excerpt_clean"]   = strip_html(post_data["excerpt"])
        enriched_records.append(post_data)

    time.sleep(0.6)

print(f"\n Successfully fetched: {len(enriched_records)} / {total} articles")

[  1/161] Fetching: kerajaan-samudera-pasai  (sejarah-smp-kelas-7)
[  2/161] Fetching: sejarah-nenek-moyang-bangsa-indonesia  (sejarah-smp-kelas-7)
[  3/161] Fetching: manusia-dan-dinosaurus  (sejarah-smp-kelas-7)
[  4/161] Fetching: perang-saudara-kerajaan-kediri  (sejarah-smp-kelas-7)
[  5/161] Fetching: masa-penjajahan  (sejarah-smp-kelas-8)
[  6/161] Fetching: kedatangan-nica-di-indonesia  (sejarah-smp-kelas-8)
[  7/161] Fetching: kedatangan-pasukan-afnei-di-indonesia  (sejarah-smp-kelas-8)
[  8/161] Fetching: tahukah-kamu-mengenai-organisasi-pergerakan-indonesia  (sejarah-smp-kelas-8)
[  9/161] Fetching: sejarah-kelahiran-masa-orde-baru  (sejarah-smp-kelas-9)
[ 10/161] Fetching: mengenal-perjanjian-renville  (sejarah-smp-kelas-9)
[ 11/161] Fetching: tokoh-kemerdekaan-dan-revolusi-indonesia  (sejarah-smp-kelas-9)
[ 12/161] Fetching: republik-indonesia-serikat  (sejarah-smp-kelas-9)
[ 13/161] Fetching: 5-bentuk-penyimpangan-demokrasi-terpimpin-terhadap-politik-luar-negeri  (sejarah-

In [ ]:
lengths = [(r["title"], len(r["content_clean"])) for r in enriched_records]
lengths.sort(key=lambda x: x[1])

print("\n  Shortest articles (check for truncation):")
for title, length in lengths[:5]:
    print(f"  {length:>6} chars — {title}")

print("\n Longest articles:")
for title, length in lengths[-5:]:
    print(f"  {length:>6} chars — {title}")


  Shortest articles (check for truncation):
    2372 chars — Pemilu Pertama di Kabinet Burhanuddin Harahap | Sejarah Kelas 12
    2549 chars — Program Kerja dan Kemunduran Kabinet Ali Sastroamidjojo 2 | Sejarah Kelas 12
    2903 chars — Mengenal Sistem Ekonomi Ali Baba | Sejarah Kelas 12
    2911 chars — Perundingan Hooge-Veluwe: Upaya Indonesia Pertahankan Kemerdekaan
    3101 chars — Kebijakan Gunting Syafruddin | Sejarah Kelas 12

 Longest articles:
   18388 chars — Mengetahui Perkembangan IPTEK di Masa Kini | Sejarah Kelas 12
   19113 chars — 7 Peran Indonesia dalam Perdamaian Dunia | Sejarah Kelas 12
   20011 chars — 10 Organisasi Pergerakan Nasional Indonesia &#038; Tokoh Pendirinya | Sejarah Kelas 11
   21042 chars — Peradaban Yunani Kuno: Sejarah, Raja, dan Peninggalannya | Sejarah Kelas 10
   22434 chars — 13 Kerajaan Maritim Hindu-Buddha di Nusantara | Sejarah Kelas 11


In [ ]:
df = pd.DataFrame(enriched_records)

col_order = [
    "id", "title", "url", "slug", "date",
    "source_category", "excerpt_clean", "content_clean",
    "content", "excerpt",
]
df = df[[c for c in col_order if c in df.columns]]

print(f"Shape: {df.shape}")
print("\nSample:")
df[["title", "url", "source_category", "date",
    "source_category", "excerpt_clean", "content_clean",
    "content", "excerpt"]].head(10)

Shape: (161, 10)

Sample:


,title,url,source_category,date,source_category,excerpt_clean,content_clean,content,excerpt
0,"Kesultanan Samudera Pasai: Sejarah, Kejayaan, ...",https://www.ruangguru.com/blog/kerajaan-samude...,sejarah-smp-kelas-7,2025-05-16T10:00:00,sejarah-smp-kelas-7,Artikel Sejarah kelas 7 ini membahas tentang s...,Artikel Sejarah kelas 7 ini membahas tentang s...,"<p><img decoding=""async"" class=""aligncenter si...",<p>Artikel Sejarah kelas 7 ini membahas tentan...
1,Mengenal Sejarah Nenek Moyang Bangsa Indonesia...,https://www.ruangguru.com/blog/sejarah-nenek-m...,sejarah-smp-kelas-7,2024-09-25T04:00:00,sejarah-smp-kelas-7,"Yuk, kita bahas lebih jauh tentang sejarah asa...","Yuk, kita bahas lebih jauh tentang sejarah asa...","<p style=""text-align: justify;""><img decoding=...","<p>Yuk, kita bahas lebih jauh tentang sejarah ..."
2,Apakah Manusia Pernah Hidup Bersama Dinosaurus...,https://www.ruangguru.com/blog/manusia-dan-din...,sejarah-smp-kelas-7,2022-02-11T09:47:23,sejarah-smp-kelas-7,Apakah manusia pernah hidup bersama dinosaurus...,Apakah manusia pernah hidup bersama dinosaurus...,"<p><img decoding=""async"" style=""width: 820px; ...",<p>Apakah manusia pernah hidup bersama dinosau...
3,"Selain Avengers, Kerajaan Kediri juga Pernah M...",https://www.ruangguru.com/blog/perang-saudara-...,sejarah-smp-kelas-7,2019-10-18T12:00:00,sejarah-smp-kelas-7,Artikel ini menceritakan tentang perang saudar...,Artikel ini menceritakan tentang perang saudar...,"<p style=""text-align: center;""><img decoding=""...",<p>Artikel ini menceritakan tentang perang sau...
4,Kehidupan Masyarakat pada Masa Penjajahan Kolo...,https://www.ruangguru.com/blog/masa-penjajahan,sejarah-smp-kelas-8,2026-01-20T13:00:08,sejarah-smp-kelas-8,Artikel Sejarah kelas 8 ini menjelaskan tentan...,Artikel Sejarah kelas 8 ini menjelaskan tentan...,"<p><img decoding=""async"" class=""aligncenter si...",<p>Artikel Sejarah kelas 8 ini menjelaskan ten...
5,Tugas dan Tujuan Kedatangan NICA di Indonesia ...,https://www.ruangguru.com/blog/kedatangan-nica...,sejarah-smp-kelas-8,2024-05-28T10:00:00,sejarah-smp-kelas-8,Artikel Sejarah kelas 8 ini menjelaskan tentan...,Artikel Sejarah kelas 8 ini menjelaskan tentan...,"<p style=""text-align: center;""><img decoding=""...",<p>Artikel Sejarah kelas 8 ini menjelaskan ten...
6,"Sejarah, Tokoh, dan Tugas Pasukan AFNEI di Ind...",https://www.ruangguru.com/blog/kedatangan-pasu...,sejarah-smp-kelas-8,2024-04-23T13:00:00,sejarah-smp-kelas-8,Tahukah kamu tentang sejarah datangnya pasukan...,Tahukah kamu tentang sejarah datangnya pasukan...,"<p style=""text-align: center;""><img decoding=""...",<p>Tahukah kamu tentang sejarah datangnya pasu...
7,"Yuk, Kenali 6 Organisasi Pergerakan Indonesia ...",https://www.ruangguru.com/blog/tahukah-kamu-me...,sejarah-smp-kelas-8,2021-03-01T02:00:00,sejarah-smp-kelas-8,Tahukah kamu mengenai organisasi pergerakan In...,Tahukah kamu mengenai organisasi pergerakan In...,"<p><img decoding=""async"" src=""https://cdn-web....",<p>Tahukah kamu mengenai organisasi pergerakan...
8,Latar Belakang Masa Orde Baru &#038; Sistem Pe...,https://www.ruangguru.com/blog/sejarah-kelahir...,sejarah-smp-kelas-9,2026-01-05T14:00:00,sejarah-smp-kelas-9,Artikel Sejarah kelas 9 ini akan membahas tent...,Artikel Sejarah kelas 9 ini akan membahas tent...,"<p style=""text-align: justify;""><img decoding=...",<p>Artikel Sejarah kelas 9 ini akan membahas t...
9,"Perjanjian Renville: Latar Belakang, Hasil &#0...",https://www.ruangguru.com/blog/mengenal-perjan...,sejarah-smp-kelas-9,2025-11-17T02:45:31,sejarah-smp-kelas-9,"Setelah merdeka, ternyata Indonesia masih haru...","Setelah merdeka, ternyata Indonesia masih haru...","<p><img decoding=""async"" class=""aligncenter si...","<p>Setelah merdeka, ternyata Indonesia masih h..."


In [ ]:
OUTPUT_FILE = "ruangguru_sejarah_kelas7_12.csv"

df_clean = df.drop(columns=["content", "excerpt"], errors="ignore")

df_clean.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL,
)

print(f" Saved {len(df_clean)} rows -> {OUTPUT_FILE}")


 Saved 161 rows → ruangguru_sejarah_kelas7_12.csv


In [ ]:
print("Articles per source_category:")
print(df["source_category"].value_counts().to_string())

print("\nDate range:")
print(f"  Oldest : {df['date'].min()}")
print(f"  Newest : {df['date'].max()}")

print("\nAvg content length (chars):")
print(f"  {df['content_clean'].str.len().mean():.0f} chars/article")

sample = df.iloc[0]
print(f"\n── Sample Article ──────────────────────────────────")
print(f"Title   : {sample['title']}")
print(f"URL     : {sample['url']}")
print(f"Category: {sample['source_category']}")
print(f"Date    : {sample['date']}")
print(f"Chars   : {len(sample['content_clean'])}")
print(f"\nFull Content:\n{sample['content_clean']}")

Articles per source_category:
source_category
sejarah-sma-kelas-11    49
sejarah-sma-kelas-12    49
sejarah-sma-kelas-10    42
sejarah-smp-kelas-9     13
sejarah-smp-kelas-8      4
sejarah-smp-kelas-7      4

Date range:
  Oldest : 2017-09-22T06:03:08
  Newest : 2026-05-19T09:35:44

Avg content length (chars):
  9032 chars/article

── Sample Article ──────────────────────────────────
Title   : Kesultanan Samudera Pasai: Sejarah, Kejayaan, dan Keruntuhan | Sejarah Kelas 7
URL     : https://www.ruangguru.com/blog/kerajaan-samudera-pasai
Category: sejarah-smp-kelas-7
Date    : 2025-05-16T10:00:00
Chars   : 5019

Full Content:
Artikel Sejarah kelas 7 ini membahas tentang sejarah, masa kejayaan, dan penyebab runtuhnya Kesultanan Samudera Pasai. —   Selain Kerajaan Hindu-Budha, ternyata Nusantara juga punya Kerajaan atau Kesultanan Islam, loh. Munculnya Kesultanan Islam di Indonesia disebabkan oleh para pedagang Islam dari Arab, India, dan Persia yang awalnya singgah untuk berdagang, lama ke